<a href="https://colab.research.google.com/github/upohaar/upohaar/blob/main/TB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'Getting started.pdf', 'video', 'inbound5716292180913652378.pdf', 'inbound5759038546891737661.pdf', 'Screenshot_2021-10-16-20-18-26-812_com.android.chrome.jpg', 'Classroom', 'roll-1373-OOP Lab report.pdf', 'Sumaiya Nobi Upohaar - Semester Final Exam Lab Report - 2023.gdoc', 'IMG_2627.MOV', '1709384038576.jpeg', 'DS LAB REPORT - 1366 (CSE-35th)-2 (1).pdf.gdoc', 'DS LAB REPORT - 1366 (CSE-35th)-2.pdf.gdoc', 'Untitled presentation (1).gslides', 'Untitled presentation.gslides', 'SQL Examples with Unique Variable Names.gdoc', 'SQL Queries and Examples.gdoc', 'SQL Queries .gdoc', '8086 অ্যাসেম্বলি কোড ব্যাখ্যা (বাংলায়).gdoc', 'task1.pdf', 'মেনু সংশোধন ও আপডেট.gsheet', 'মেনু সংশোধন ও ফরম্যাটিং.gsheet', 'gbeeepc.png', 'Untitled spreadsheet.gsheet', 'Recording_10.m4a', 'Recording_15.m4a', 'Recording_13.m4a', 'Recording_7.m4a', 'Recording_6.m4a', 'dataset.zip']


In [3]:
!unzip -q "/content/drive/MyDrive/dataset.zip" -d /content/


In [4]:
import os

print("dataset folder:", os.listdir("dataset"))
print("Normal:", len(os.listdir("dataset/Normal")))
print("Tuberculosis:", len(os.listdir("dataset/Tuberculosis")))

dataset folder: ['Tuberculosis', 'Normal']
Normal: 3500
Tuberculosis: 700


In [5]:
!nvidia-smi


Wed Oct  7 20:10:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import os
print(os.listdir("dataset"))



['Tuberculosis', 'Normal']


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory

IMG_SIZE = (224, 224)
BATCH = 32

train_ds = image_dataset_from_directory(
    "dataset",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    validation_split=0.2,
    subset="training",
    seed=123
)

val_ds = image_dataset_from_directory(
    "dataset",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    validation_split=0.2,
    subset="validation",
    seed=123
)

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(train_ds, validation_data=val_ds, epochs=10)

model.save("/content/drive/MyDrive/tb_model.h5")


Found 4200 files belonging to 2 classes.
Using 3360 files for training.
Found 4200 files belonging to 2 classes.
Using 840 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 43s 282ms/step - accuracy: 0.8747 - loss: 0.2984 - val_accuracy: 0.9286 - val_loss: 0.1868
Epoch 2/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 23s 221ms/step - accuracy: 0.9387 - loss: 0.1650 - val_accuracy: 0.9571 - val_loss: 0.1305
Epoch 3/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 22s 207ms/step - accuracy: 0.9595 - loss: 0.1248 - val_accuracy: 0.9690 - val_loss: 0.1056
Epoch 4/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 24s 224ms/step - accuracy: 0.9646 - loss: 0.1090 - val_accuracy: 0.9702 - val_loss: 0.0930
Epoch 5/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 39s 205ms/step - accuracy: 0.9693 - loss: 0.0951 - val_accuracy: 0.9714 - val_loss: 0.0859
Epoch 6/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 28s 263ms/step - accuracy: 0.9741 - loss: 0.0874 - val_accuracy: 0.9679 - val_loss: 0.0816
Epoch 7/10
105/105 ━

In [8]:
import os
files = os.listdir("/content/drive/MyDrive")
print("tb_model.h5" in files)

True


In [9]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend((preds > 0.5).astype(int).flatten())

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Normal", "Tuberculosis"]))

Confusion Matrix:
[[690   3]
 [ 14 133]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.98      1.00      0.99       693
Tuberculosis       0.98      0.90      0.94       147

    accuracy                           0.98       840
   macro avg       0.98      0.95      0.96       840
weighted avg       0.98      0.98      0.98       840



In [10]:

y_pred = (preds > 0.3).astype(int).flatten()

In [11]:
class_weight = {0: 1.0, 1: 5.0}  # TB কে বেশি গুরুত্ব

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight
)

Epoch 1/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 29s 207ms/step - accuracy: 0.9699 - loss: 0.1879 - val_accuracy: 0.9774 - val_loss: 0.0809
Epoch 2/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 21s 199ms/step - accuracy: 0.9786 - loss: 0.1585 - val_accuracy: 0.9833 - val_loss: 0.0641
Epoch 3/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 23s 219ms/step - accuracy: 0.9753 - loss: 0.1457 - val_accuracy: 0.9798 - val_loss: 0.0690
Epoch 4/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 22s 206ms/step - accuracy: 0.9732 - loss: 0.1487 - val_accuracy: 0.9833 - val_loss: 0.0573
Epoch 5/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 47s 262ms/step - accuracy: 0.9723 - loss: 0.1430 - val_accuracy: 0.9821 - val_loss: 0.0640
Epoch 6/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 22s 212ms/step - accuracy: 0.9774 - loss: 0.1294 - val_accuracy: 0.9833 - val_loss: 0.0532
Epoch 7/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 23s 216ms/step - accuracy: 0.9792 - loss: 0.1194 - val_accuracy: 0.9821 - val_loss: 0.0546
Epoch 8/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 39s 200ms/step - accuracy: 0.9792 - loss: 0

In [12]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend((preds > 0.5).astype(int).flatten())

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Normal", "Tuberculosis"]))

Confusion Matrix:
[[682  11]
 [  4 143]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.98      0.99       693
Tuberculosis       0.93      0.97      0.95       147

    accuracy                           0.98       840
   macro avg       0.96      0.98      0.97       840
weighted avg       0.98      0.98      0.98       840



In [13]:
model.save("/content/drive/MyDrive/tb_model_improved.keras")

In [14]:
import os

# Drive এ চেক করো
print("Drive এ:")
for f in os.listdir("/content/drive/MyDrive"):
    if "tb_model" in f:
        print("  ", f)

# Colab এর লোকাল স্টোরেজে চেক করো
print("\nColab এ:")
for f in os.listdir("/content"):
    if "tb_model" in f:
        print("  ", f)

Drive এ:
   tb_model.h5
   tb_model_improved.keras

Colab এ:


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory

IMG_SIZE = (224, 224)
BATCH = 32

train_ds = image_dataset_from_directory(
    "dataset",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    validation_split=0.2,
    subset="training",
    seed=123
)

val_ds = image_dataset_from_directory(
    "dataset",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    validation_split=0.2,
    subset="validation",
    seed=123
)

# Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

model = models.Sequential([
    data_augmentation,
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

class_weight = {0: 1.0, 1: 5.0}

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight
)

Found 4200 files belonging to 2 classes.
Using 3360 files for training.
Found 4200 files belonging to 2 classes.
Using 840 files for validation.
Epoch 1/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 37s 241ms/step - accuracy: 0.7336 - loss: 0.8986 - val_accuracy: 0.9131 - val_loss: 0.2746
Epoch 2/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 23s 219ms/step - accuracy: 0.8372 - loss: 0.6421 - val_accuracy: 0.9202 - val_loss: 0.2412
Epoch 3/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 41s 221ms/step - accuracy: 0.8613 - loss: 0.5539 - val_accuracy: 0.9155 - val_loss: 0.2159
Epoch 4/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 41s 225ms/step - accuracy: 0.8664 - loss: 0.5352 - val_accuracy: 0.9107 - val_loss: 0.2120
Epoch 5/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 22s 205ms/step - accuracy: 0.8813 - loss: 0.5017 - val_accuracy: 0.9310 - val_loss: 0.1963
Epoch 6/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 28s 272ms/step - accuracy: 0.8881 - loss: 0.4513 - val_accuracy: 0.9250 - val_loss: 0.1899
Epoch 7/10
 59/105 ━━━━━━━━━━━━━━━━━━━━ 8s 183ms/step - accuracy: 0.8